## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 48 samples and 9986 columns.
First few rows of the data:


,Months,Survival Status,sample_id,Tumor exome data,Normal exome data a,Tumor RNAseq data,Normal RNA seq,Race,Dx Date,Survival Date,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,63.4,0.0,5,X,NaN,X,NaN,Chinese,2004-02-23,2009-05-10,...,7.953216,6.880651,8.201768,8.172487,6.538856,7.478783,9.094757,8.576978,7.572553,8.143199
1,8.4,1.0,6,X,NaN,X,NaN,Chinese,2004-03-10,2004-11-17,...,7.694184,6.722906,7.988022,7.789258,5.651215,6.951462,8.818602,8.100091,6.968440,7.802846
2,22.7,1.0,15,X,X,X,NaN,Chinese,2005-08-29,2007-07-10,...,5.728599,4.288251,5.573461,5.697862,3.887059,4.919057,6.646728,6.397195,5.485431,5.815798
3,14.6,1.0,18,X,X,X,NaN,Chinese,2006-05-08,2007-07-19,...,5.639418,4.762175,5.660045,5.970535,4.085786,5.140975,6.639852,7.412492,7.142715,5.853071
4,23.1,0.0,20,X,X,X,NaN,Chinese,2006-10-07,2008-08-29,...,6.201917,4.753634,6.057103,6.255138,4.379792,5.403289,7.126437,6.983460,5.742906,6.198644


## Load gene sets

In [2]:
MSIGDB_JSON = Path("../../resources/h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'sample_title'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
